# MAna Recommenders Module

`MAna.recommend` documents the practical recommender-system pieces inside
MAna: interaction validation, sparse user-item matrices, popularity baselines,
content recommendations, item-neighborhood collaborative filtering, hybrid
fusion, and offline ranking evaluation.

This notebook uses real data from an old recommender project:

- `ratings_small.csv`, a MovieLens-style user/movie/rating/timestamp table.
- A compact movie catalog joined from TMDB metadata, keywords, and MovieLens
  links.

The examples avoid synthetic users. When a user profile is needed, the notebook
selects a real eligible user from the training interactions.

## 1. Setup

The notebook uses MAna's data module for loading, MAna's NLP module for text
features, and MAna's recommender module for the recommender logic.

In [1]:
from pathlib import Path
import ast

import numpy as np
import pandas as pd

from MAna.data import DataCleaner, read_data
from MAna.nlp import TextCleaner, TextVectorizer
from MAna.recommend import (
    BaseRecommender,
    ContentRecommender,
    HybridRecommender,
    InteractionMatrix,
    ItemKNNRecommender,
    NumericFeatureTransformer,
    PopularityRecommender,
    RankedItem,
    build_interaction_matrix,
    build_user_histories,
    catalog_coverage,
    combine_text_features,
    concatenate_feature_embeddings,
    evaluate_rankings,
    evaluate_recommender,
    fuse_rankings,
    intra_list_diversity,
    prepare_numeric_features,
    recommendation_frame,
    reciprocal_rank_fusion,
    temporal_train_test_split,
    validate_interactions,
    weighted_score_fusion,
)

pd.set_option("display.max_colwidth", 120)

DATA_DIR = Path("data")
if not DATA_DIR.exists():
    DATA_DIR = Path("docs/notebooks/data")
RATINGS_PATH = DATA_DIR / "movielens_ratings_small.csv"
CATALOG_PATH = DATA_DIR / "movielens_movie_catalog.csv"

## 2. Load Real Interaction and Catalog Data

The recommender module expects two common shapes:

- An interaction table: `user_id`, `item_id`, optional score, optional time.
- A catalog table: one row per item with text/numeric metadata.

In [2]:
ratings_raw = read_data(RATINGS_PATH)
catalog_raw = read_data(CATALOG_PATH)

{
    "ratings_shape": ratings_raw.shape,
    "catalog_shape": catalog_raw.shape,
    "ratings_columns": ratings_raw.columns.tolist(),
    "catalog_columns": catalog_raw.columns.tolist(),
}

{'ratings_shape': (100004, 4),
 'catalog_shape': (9066, 13),
 'ratings_columns': ['userId', 'movieId', 'rating', 'timestamp'],
 'catalog_columns': ['movieId',
  'tmdbId',
  'imdbId',
  'title',
  'overview',
  'genres',
  'keywords',
  'vote_average',
  'vote_count',
  'popularity',
  'release_date',
  'runtime',
  'original_language']}

In [3]:
# MAna.data adapts the source schema to the generic user-item contract,
# removes exact duplicates, coerces scores, and parses event time as UTC.
ratings = (
    DataCleaner(ratings_raw, verbose=False)
    .rename_columns({"userId": "user_id", "movieId": "item_id"})
    .remove_duplicates()
    .coerce_numeric(columns=["user_id", "item_id", "rating"])
    .parse_dates(columns=["timestamp"], extract_features=False, utc=True)
    .drop_missing_rows(subset=["user_id", "item_id", "rating", "timestamp"])
    .get_cleaned_data()
)

catalog = (
    DataCleaner(catalog_raw, verbose=False)
    .rename_columns({"movieId": "item_id"})
    .coerce_numeric(columns=["item_id"])
    .drop_missing_rows(subset=["item_id"])
    .remove_duplicates(subset=["item_id"])
    .get_cleaned_data()
)

{
    "ratings": ratings.shape,
    "users": ratings["user_id"].nunique(),
    "rated_items": ratings["item_id"].nunique(),
    "catalog_items": catalog["item_id"].nunique(),
    "rating_range": (ratings["rating"].min(), ratings["rating"].max()),
}

{'ratings': (100004, 4),
 'users': 671,
 'rated_items': 9066,
 'catalog_items': 9066,
 'rating_range': (0.5, 5.0)}

In [4]:
# validate_interactions() checks the core recommender contract without mutating
# the input frame. It also converts scores and timestamps to reliable types.
validated_ratings = validate_interactions(
    ratings,
    user_column="user_id",
    item_column="item_id",
    score_column="rating",
    timestamp_column="timestamp",
)

validated_ratings.dtypes

user_id                    int64
item_id                    int64
rating                   float64
timestamp    datetime64[ns, UTC]
dtype: object

In [5]:
# Keep a title lookup around so recommendation DataFrames are readable.
item_lookup = catalog.set_index("item_id")[
    ["title", "genres", "vote_average", "vote_count"]
].to_dict("index")


def add_movie_titles(frame):
    local = frame.copy()
    local["title"] = local["item_id"].map(
        lambda item_id: item_lookup.get(item_id, {}).get("title")
    )
    return local


ratings.head()

,user_id,item_id,rating,timestamp
0,1,31,2.5,2009-12-14 02:52:24+00:00
1,1,1029,3.0,2009-12-14 02:52:59+00:00
2,1,1061,3.0,2009-12-14 02:53:02+00:00
3,1,1129,2.0,2009-12-14 02:53:05+00:00
4,1,1172,4.0,2009-12-14 02:53:25+00:00


## 3. Temporal Splitting and Interaction Matrices

Recommender evaluation can leak future behavior if the split is random. MAna's
temporal split holds out each eligible user's latest interaction.

In [6]:
train, test = temporal_train_test_split(
    ratings,
    user_column="user_id",
    item_column="item_id",
    score_column="rating",
    timestamp_column="timestamp",
    holdout=1,
    minimum_interactions=20,
)

{
    "train_shape": train.shape,
    "test_shape": test.shape,
    "test_users": test["user_id"].nunique(),
}

{'train_shape': (99333, 4), 'test_shape': (671, 4), 'test_users': 671}

In [7]:
# The check below is the reason we use temporal splitting: for each held-out
# user, the test item happened after the training items.
split_audit = (
    train.groupby("user_id")["timestamp"].max().rename("latest_train")
    .to_frame()
    .join(test.groupby("user_id")["timestamp"].min().rename("earliest_test"))
    .dropna()
)
split_audit["is_future_holdout"] = split_audit["latest_train"] < split_audit["earliest_test"]

split_audit["is_future_holdout"].value_counts()

is_future_holdout
True     544
False    127
Name: count, dtype: int64

In [8]:
train.head()

,user_id,item_id,rating,timestamp
0,1,31,2.5,2009-12-14 02:52:24+00:00
1,1,1029,3.0,2009-12-14 02:52:59+00:00
2,1,1061,3.0,2009-12-14 02:53:02+00:00
3,1,1129,2.0,2009-12-14 02:53:05+00:00
4,1,1263,2.0,2009-12-14 02:52:31+00:00


In [9]:
# build_interaction_matrix() creates a sparse user-item matrix while preserving
# stable user/item id mappings. Aggregation decides how duplicate ratings are
# combined.
interaction_matrix = build_interaction_matrix(
    train,
    user_column="user_id",
    item_column="item_id",
    score_column="rating",
    aggregation="mean",
)

{
    "matrix_shape": interaction_matrix.matrix.shape,
    "nonzero_interactions": interaction_matrix.matrix.nnz,
    "first_user": interaction_matrix.user_ids[0],
    "first_item": interaction_matrix.item_ids[0],
}

{'matrix_shape': (671, 9031),
 'nonzero_interactions': 99333,
 'first_user': 1,
 'first_item': 31}

In [10]:
# build_user_histories() converts interactions into ordered item histories.
# We build two histories: all seen items for diagnostics, and high-rated items
# for content profiles.
all_histories = build_user_histories(
    train,
    user_column="user_id",
    item_column="item_id",
    score_column="rating",
    timestamp_column="timestamp",
    minimum_interactions=1,
)

positive_histories = build_user_histories(
    train,
    user_column="user_id",
    item_column="item_id",
    score_column="rating",
    timestamp_column="timestamp",
    minimum_interactions=2,
    minimum_score=4.0,
)

example_user = max(positive_histories, key=lambda user: len(positive_histories[user]))

{
    "users_with_any_history": len(all_histories),
    "users_with_positive_profiles": len(positive_histories),
    "example_user": example_user,
    "example_positive_items": len(positive_histories[example_user]),
}

{'users_with_any_history': 671,
 'users_with_positive_profiles': 667,
 'example_user': 564,
 'example_positive_items': 1115}

## 4. Shared Result Formatting

`recommendation_frame()` is the common output contract: `item_id`, `score`,
`rank`, and `source`, with optional metadata.

In [11]:
demo_frame = recommendation_frame(
    item_ids=[1, 1, 2, 3],
    scores=[0.5, 0.7, 0.6, 0.4],
    top_n=3,
    source="manual_demo",
    metadata={item_id: item_lookup.get(item_id, {}) for item_id in [1, 2, 3]},
)

add_movie_titles(demo_frame)

,item_id,score,rank,source,metadata,title
0,1,0.7,1,manual_demo,"{'title': 'Toy Story', 'genres': '[{'id': 16, 'name': 'Animation'}, {'id': 35, 'name': 'Comedy'}, {'id': 10751, 'nam...",Toy Story
1,2,0.6,2,manual_demo,"{'title': 'Jumanji', 'genres': '[{'id': 12, 'name': 'Adventure'}, {'id': 14, 'name': 'Fantasy'}, {'id': 10751, 'name...",Jumanji
2,3,0.4,3,manual_demo,"{'title': 'Grumpier Old Men', 'genres': '[{'id': 10749, 'name': 'Romance'}, {'id': 35, 'name': 'Comedy'}]', 'vote_av...",Grumpier Old Men


## 5. Popularity Recommenders

Popularity is not a throwaway baseline. It handles cold-start users, gives a
sanity check for fancier models, and often performs surprisingly well.

In [12]:
# Count mode uses interaction volume. This is useful when clicks/views are the
# only signal, or when ratings are not trustworthy.
count_popularity = PopularityRecommender(
    user_column="user_id",
    item_column="item_id",
    score_mode="count",
).fit(train)

add_movie_titles(count_popularity.recommend(user_id=None, top_n=8))

,item_id,score,rank,source,interaction_count,weighted_count,title
0,356,339.0,1,popularity,339,339.0,Forrest Gump
1,296,322.0,2,popularity,322,322.0,Pulp Fiction
2,318,308.0,3,popularity,308,308.0,The Shawshank Redemption
3,593,302.0,4,popularity,302,302.0,The Silence of the Lambs
4,260,291.0,5,popularity,291,291.0,Star Wars
5,480,273.0,6,popularity,273,273.0,Jurassic Park
6,2571,259.0,7,popularity,259,259.0,The Matrix
7,1,246.0,8,popularity,246,246.0,Toy Story


In [13]:
example_user

564

In [14]:
# Bayesian mode shrinks high-rated but low-count movies toward the global mean.
# Recency decay lets newer interactions matter more without discarding history.
bayesian_popularity = PopularityRecommender(
    user_column="user_id",
    item_column="item_id",
    score_column="rating",
    timestamp_column="timestamp",
    score_mode="bayesian",
    shrinkage=25,
    half_life_days=365 * 4,
).fit(train)

add_movie_titles(bayesian_popularity.recommend(example_user, top_n=8))

,item_id,score,rank,source,interaction_count,weighted_count,mean_score,title
0,318,4.244512,1,popularity,308,84.048373,4.463542,The Shawshank Redemption
1,6016,3.979637,2,popularity,68,30.393270,4.367461,City of God
2,4993,3.962391,3,popularity,200,79.341321,4.105521,The Lord of the Rings: The Fellowship of the Ring
3,58559,3.948992,4,popularity,116,66.843315,4.113872,The Dark Knight
4,4226,3.921775,5,popularity,132,44.089639,4.156313,Memento
5,48516,3.911149,6,popularity,82,40.841279,4.157837,The Departed
6,7153,3.903359,7,popularity,176,73.599267,4.037604,The Lord of the Rings: The Return of the King
7,79132,3.891420,8,popularity,109,73.200087,4.022319,Inception


In [15]:
# exclude_seen=True removes the user's own history. Turning it off is useful
# when you want to audit raw global rank, not serve a personalized list.
seen_vs_not_seen = {
    "exclude_seen": add_movie_titles(
        bayesian_popularity.recommend(example_user, top_n=5, exclude_seen=True)
    ),
    "include_seen": add_movie_titles(
        bayesian_popularity.recommend(example_user, top_n=5, exclude_seen=False)
    ),
}

seen_vs_not_seen["exclude_seen"]

,item_id,score,rank,source,interaction_count,weighted_count,mean_score,title
0,318,4.244512,1,popularity,308,84.048373,4.463542,The Shawshank Redemption
1,6016,3.979637,2,popularity,68,30.393270,4.367461,City of God
2,4993,3.962391,3,popularity,200,79.341321,4.105521,The Lord of the Rings: The Fellowship of the Ring
3,58559,3.948992,4,popularity,116,66.843315,4.113872,The Dark Knight
4,4226,3.921775,5,popularity,132,44.089639,4.156313,Memento


## 6. Content Feature Engineering

Content recommenders need item features. MAna provides helpers for combining
text fields, preparing numeric fields, and concatenating feature blocks.

In [16]:
def extract_names(value):
    # The source metadata stores genres and keywords as stringified lists of
    # dictionaries. This parser keeps the notebook robust to missing/bad rows.
    try:
        items = ast.literal_eval(value) if isinstance(value, str) else []
    except (ValueError, SyntaxError):
        items = []
    return " ".join(
        item.get("name", "")
        for item in items
        if isinstance(item, dict) and item.get("name")
    )


catalog = DataCleaner(catalog, verbose=False).fix_missing_values(
    fill_value={"genres": "[]", "keywords": "[]"}
).get_cleaned_data()
catalog["genres_text"] = catalog["genres"].apply(extract_names)
catalog["keywords_text"] = catalog["keywords"].apply(extract_names)

catalog[["item_id", "title", "genres_text", "keywords_text"]].head()

,item_id,title,genres_text,keywords_text
0,1,Toy Story,Animation Comedy Family,jealousy toy boy friendship friends rivalry boy next door new toy toy comes to life
1,2,Jumanji,Adventure Fantasy Family,board game disappearance based on children's book new home recluse giant insect
2,3,Grumpier Old Men,Romance Comedy,fishing best friend duringcreditsstinger old men
3,4,Waiting to Exhale,Comedy Drama Romance,based on novel interracial relationship single mother divorce chick flick
4,5,Father of the Bride Part II,Comedy,baby midlife crisis confidence aging daughter mother daughter relationship pregnancy contraception gynecologist


In [17]:
# combine_text_features() lets important fields repeat more often. Here titles
# and genres get extra weight because they are short and meaningful.
catalog["content_text"] = combine_text_features(
    catalog,
    ["title", "overview", "genres_text", "keywords_text"],
    weights={"title": 2, "genres_text": 2},
)

catalog[["title", "content_text"]].head(3)

,title,content_text
0,Toy Story,"Toy Story Toy Story Led by Woody, Andy's toys live happily in his room until Andy's birthday brings Buzz Lightyear o..."
1,Jumanji,Jumanji Jumanji When siblings Judy and Peter discover an enchanted board game that opens the door to a magical world...
2,Grumpier Old Men,Grumpier Old Men Grumpier Old Men A family wedding reignites the ancient feud between next-door neighbors and fishin...


In [18]:
# TextVectorizer comes from MAna.nlp. The recommender module does not force a
# single text representation, so you can bring TF-IDF, embeddings, or your own
# feature matrix.
content_vectorizer = TextVectorizer(
    method="tfidf",
    cleaner=TextCleaner(stop_words="english", min_token_length=2),
    max_features=800,
    ngram_range=(1, 2),
    min_df=3,
)

text_features = content_vectorizer.fit_transform(catalog["content_text"]).matrix

{
    "text_feature_shape": text_features.shape,
    "sample_terms": content_vectorizer.get_feature_names_out()[:15].tolist(),
}

{'text_feature_shape': (9066, 800),
 'sample_terms': ['abuse',
  'accident',
  'action',
  'action adventure',
  'action comedy',
  'action crime',
  'action drama',
  'action science',
  'action thriller',
  'actor',
  'actress',
  'adult',
  'adventure',
  'adventure action',
  'adventure animation']}

In [19]:
# prepare_numeric_features() median-imputes and optionally standardizes numeric
# metadata. The returned transformer can be reused on future catalog rows.
numeric_features, numeric_transformer = prepare_numeric_features(
    catalog,
    ["vote_average", "vote_count", "runtime"],
)

future_numeric = numeric_transformer.transform(
    catalog[["vote_average", "vote_count", "runtime"]].head(3)
)

{
    "numeric_feature_shape": numeric_features.shape,
    "future_transform_shape": future_numeric.shape,
}

{'numeric_feature_shape': (9066, 3), 'future_transform_shape': (3, 3)}

In [20]:
# concatenate_feature_embeddings() expects aligned dense blocks. The catalog is
# small enough for a dense TF-IDF inspection matrix here; for very large catalogs
# you could keep sparse features and pass them directly to ContentRecommender.
content_features = concatenate_feature_embeddings(
    text_features.toarray().astype("float32"),
    numeric_features,
    weights=[1.0, 0.15],
)

content_features.shape

(9066, 803)

## 7. Content Recommender

`ContentRecommender` can recommend similar items, recommend from explicit liked
items, or recommend for users when you provide user histories.

In [21]:
content_recommender = ContentRecommender().fit(
    catalog["item_id"],
    content_features,
    metadata=catalog[["title", "genres_text", "vote_average", "vote_count"]],
    user_histories=positive_histories,
)

add_movie_titles(content_recommender.similar_items(1, top_n=8))

,item_id,score,rank,source,metadata,title
0,3114,0.686679,1,content,"{'title': 'Toy Story 2', 'genres_text': 'Animation Comedy Family', 'vote_average': 7.3, 'vote_count': 3914.0}",Toy Story 2
1,68954,0.668696,2,content,"{'title': 'Up', 'genres_text': 'Animation Comedy Family Adventure', 'vote_average': 7.8, 'vote_count': 7048.0}",Up
2,103335,0.640361,3,content,"{'title': 'Despicable Me 2', 'genres_text': 'Animation Comedy Family', 'vote_average': 7.0, 'vote_count': 4729.0}",Despicable Me 2
3,157296,0.617083,4,content,"{'title': 'Finding Dory', 'genres_text': 'Adventure Animation Comedy Family', 'vote_average': 6.8, 'vote_count': 433...",Finding Dory
4,4886,0.593624,5,content,"{'title': 'Monsters, Inc.', 'genres_text': 'Animation Comedy Family', 'vote_average': 7.5, 'vote_count': 6150.0}","Monsters, Inc."
5,2355,0.543981,6,content,"{'title': 'A Bug's Life', 'genres_text': 'Adventure Animation Comedy Family', 'vote_average': 6.8, 'vote_count': 237...",A Bug's Life
6,122904,0.542203,7,content,"{'title': 'Deadpool', 'genres_text': 'Action Adventure Comedy', 'vote_average': 7.4, 'vote_count': 11444.0}",Deadpool
7,4306,0.541076,8,content,"{'title': 'Shrek', 'genres_text': 'Adventure Animation Comedy Family Fantasy', 'vote_average': 7.3, 'vote_count': 41...",Shrek


In [22]:
# Build a profile from explicit likes/dislikes. We use real item IDs:
# 1 is Toy Story; 2 is Jumanji. The disliked item changes the profile direction.
explicit_profile_recs = content_recommender.recommend_from_items(
    liked_items=[1, 3114],
    disliked_items=[2],
    dislike_weight=0.35,
    top_n=8,
)

add_movie_titles(explicit_profile_recs)

,item_id,score,rank,source,metadata,title
0,103335,0.613892,1,content,"{'title': 'Despicable Me 2', 'genres_text': 'Animation Comedy Family', 'vote_average': 7.0, 'vote_count': 4729.0}",Despicable Me 2
1,68954,0.611814,2,content,"{'title': 'Up', 'genres_text': 'Animation Comedy Family Adventure', 'vote_average': 7.8, 'vote_count': 7048.0}",Up
2,157296,0.565034,3,content,"{'title': 'Finding Dory', 'genres_text': 'Adventure Animation Comedy Family', 'vote_average': 6.8, 'vote_count': 433...",Finding Dory
3,54272,0.548762,4,content,"{'title': 'The Simpsons Movie', 'genres_text': 'Animation Comedy Family', 'vote_average': 6.9, 'vote_count': 2335.0}",The Simpsons Movie
4,2355,0.528862,5,content,"{'title': 'A Bug's Life', 'genres_text': 'Adventure Animation Comedy Family', 'vote_average': 6.8, 'vote_count': 237...",A Bug's Life
5,4886,0.516910,6,content,"{'title': 'Monsters, Inc.', 'genres_text': 'Animation Comedy Family', 'vote_average': 7.5, 'vote_count': 6150.0}","Monsters, Inc."
6,4306,0.501735,7,content,"{'title': 'Shrek', 'genres_text': 'Adventure Animation Comedy Family Fantasy', 'vote_average': 7.3, 'vote_count': 41...",Shrek
7,2085,0.480323,8,content,"{'title': 'One Hundred and One Dalmatians', 'genres_text': 'Adventure Animation Comedy Family', 'vote_average': 6.8,...",One Hundred and One Dalmatians


In [23]:
# User recommendations use the real positive history selected earlier.
content_user_recs = content_recommender.recommend(example_user, top_n=8)
add_movie_titles(content_user_recs)

,item_id,score,rank,source,metadata,title
0,41769,0.419772,1,content,"{'title': 'Mozart and the Whale', 'genres_text': 'Comedy Drama Romance', 'vote_average': 6.5, 'vote_count': 41.0}",Mozart and the Whale
1,4555,0.414292,2,content,"{'title': 'Torch Song Trilogy', 'genres_text': 'Comedy Drama Romance', 'vote_average': 7.0, 'vote_count': 25.0}",Torch Song Trilogy
2,232,0.405980,3,content,"{'title': 'Eat Drink Man Woman', 'genres_text': 'Comedy Drama Romance', 'vote_average': 7.5, 'vote_count': 76.0}",Eat Drink Man Woman
3,55757,0.405425,4,content,"{'title': 'Chilly Scenes of Winter', 'genres_text': 'Comedy Drama Romance', 'vote_average': 10.0, 'vote_count': 1.0}",Chilly Scenes of Winter
4,5450,0.405341,5,content,"{'title': 'Lovely & Amazing', 'genres_text': 'Comedy Drama Romance', 'vote_average': 6.3, 'vote_count': 23.0}",Lovely & Amazing
5,1441,0.400851,6,content,"{'title': 'Benny & Joon', 'genres_text': 'Comedy Drama Romance', 'vote_average': 6.9, 'vote_count': 220.0}",Benny & Joon
6,3559,0.399977,7,content,"{'title': 'Limelight', 'genres_text': 'Comedy Drama Romance', 'vote_average': 8.0, 'vote_count': 99.0}",Limelight
7,4687,0.395685,8,content,"{'title': 'Billy Liar', 'genres_text': 'Romance Comedy Drama', 'vote_average': 6.7, 'vote_count': 34.0}",Billy Liar


In [24]:
# profile() returns the actual vector used for profile-based scoring. This is
# useful when you want to cache profiles or inspect vector dimensions.
profile_vector = content_recommender.profile(positive_histories[example_user][:10])

{
    "profile_width": profile_vector.shape[0],
    "profile_norm": float(np.linalg.norm(profile_vector)),
}

{'profile_width': 803, 'profile_norm': 1.0}

## 8. Collaborative Filtering

`ItemKNNRecommender` learns item similarity from co-occurrence in the user-item
matrix. It does not need titles, genres, or overviews.

In [25]:
item_knn = ItemKNNRecommender(
    user_column="user_id",
    item_column="item_id",
    score_column="rating",
    aggregation="mean",
    n_neighbors=60,
    minimum_similarity=0.0,
).fit(train)

add_movie_titles(item_knn.similar_items(1, top_n=8))

,item_id,score,rank,source,title
0,3114,0.601969,1,item_knn,Toy Story 2
1,260,0.577088,2,item_knn,Star Wars
2,780,0.566279,3,item_knn,Independence Day
3,356,0.561001,4,item_knn,Forrest Gump
4,1265,0.551576,5,item_knn,Groundhog Day
5,480,0.533601,6,item_knn,Jurassic Park
6,1270,0.532881,7,item_knn,Back to the Future
7,4306,0.529777,8,item_knn,Shrek


In [26]:
knn_user_recs = item_knn.recommend(example_user, top_n=8)
add_movie_titles(knn_user_recs)

,item_id,score,rank,source,title
0,1265,236.434150,1,item_knn,Groundhog Day
1,2243,174.454417,2,item_knn,Broadcast News
2,1307,173.365222,3,item_knn,When Harry Met Sally...
3,318,129.029871,4,item_knn,The Shawshank Redemption
4,1393,109.179042,5,item_knn,Jerry Maguire
5,2065,107.845009,6,item_knn,The Purple Rose of Cairo
6,913,106.784595,7,item_knn,The Maltese Falcon
7,3201,105.379150,8,item_knn,Five Easy Pieces


## 9. Hybrid Recommendation

Hybrid recommenders combine multiple candidate lists. MAna supports weighted
score fusion and reciprocal-rank fusion.

In [27]:
# weighted_score_fusion() is the low-level function. It normalizes each source
# before combining scores, so a content cosine score and a KNN score can share a
# final ranking.
manual_weighted = weighted_score_fusion(
    {
        "popularity": bayesian_popularity.recommend(example_user, top_n=20),
        "content": content_user_recs,
        "knn": knn_user_recs,
    },
    weights={"popularity": 0.6, "content": 1.0, "knn": 1.2},
    normalization="rank",
    top_n=8,
)

add_movie_titles(manual_weighted)

,item_id,score,rank,source,component_scores,title
0,1265,0.428571,1,hybrid,{'knn': 1.0},Groundhog Day
1,41769,0.357143,2,hybrid,{'content': 1.0},Mozart and the Whale
2,318,0.321429,3,hybrid,"{'popularity': 1.0, 'knn': 0.25}",The Shawshank Redemption
3,2243,0.214286,4,hybrid,{'knn': 0.5},Broadcast News
4,4555,0.178571,5,hybrid,{'content': 0.5},Torch Song Trilogy
5,1307,0.142857,6,hybrid,{'knn': 0.3333333333333333},When Harry Met Sally...
6,232,0.119048,7,hybrid,{'content': 0.3333333333333333},Eat Drink Man Woman
7,6016,0.107143,8,hybrid,{'popularity': 0.5},City of God


In [28]:
# HybridRecommender wraps fitted recommenders that share MAna's recommend()
# contract. It can use weighted scores or RAG-style reciprocal rank fusion.
weighted_hybrid = HybridRecommender(
    {
        "popularity": bayesian_popularity,
        "content": content_recommender,
        "knn": item_knn,
    },
    weights={"popularity": 0.6, "content": 1.0, "knn": 1.2},
    method="weighted",
    normalization="rank",
)

hybrid_weighted_recs = weighted_hybrid.recommend(example_user, top_n=8)
add_movie_titles(hybrid_weighted_recs)

,item_id,score,rank,source,component_scores,title
0,1265,0.428571,1,hybrid,{'knn': 1.0},Groundhog Day
1,41769,0.357143,2,hybrid,{'content': 1.0},Mozart and the Whale
2,318,0.321429,3,hybrid,"{'popularity': 1.0, 'knn': 0.25}",The Shawshank Redemption
3,2243,0.214286,4,hybrid,{'knn': 0.5},Broadcast News
4,4555,0.178571,5,hybrid,{'content': 0.5},Torch Song Trilogy
5,1307,0.142857,6,hybrid,{'knn': 0.3333333333333333},When Harry Met Sally...
6,232,0.119048,7,hybrid,{'content': 0.3333333333333333},Eat Drink Man Woman
7,6016,0.107143,8,hybrid,{'popularity': 0.5},City of God


In [29]:
rrf_hybrid = HybridRecommender(
    {
        "popularity": bayesian_popularity,
        "content": content_recommender,
        "knn": item_knn,
    },
    weights={"popularity": 0.6, "content": 1.0, "knn": 1.2},
    method="rrf",
    rank_constant=40,
)

hybrid_rrf_recs = rrf_hybrid.recommend(example_user, top_n=8)
add_movie_titles(hybrid_rrf_recs)

,item_id,score,rank,source,component_ranks,title
0,318,0.041907,1,hybrid_rrf,"{'popularity': 1, 'knn': 4}",The Shawshank Redemption
1,4993,0.033953,2,hybrid_rrf,"{'popularity': 3, 'knn': 20}",The Lord of the Rings: The Fellowship of the Ring
2,1265,0.029268,3,hybrid_rrf,{'knn': 1},Groundhog Day
3,4226,0.028918,4,hybrid_rrf,"{'popularity': 5, 'knn': 37}",Memento
4,2243,0.028571,5,hybrid_rrf,{'knn': 2},Broadcast News
5,1307,0.027907,6,hybrid_rrf,{'knn': 3},When Harry Met Sally...
6,1393,0.026667,7,hybrid_rrf,{'knn': 5},Jerry Maguire
7,5952,0.026316,8,hybrid_rrf,"{'popularity': 17, 'knn': 36}",The Lord of the Rings: The Two Towers


In [30]:
# The same rank-fusion utilities are also exported directly for custom systems.
source_rankings = {
    "content": content_user_recs["item_id"].tolist(),
    "knn": knn_user_recs["item_id"].tolist(),
}

{
    "rrf_ids": reciprocal_rank_fusion(source_rankings, top_k=5),
    "rich_rrf": fuse_rankings(source_rankings, top_k=3),
}

{'rrf_ids': [1265, 41769, 2243, 4555, 1307],
 'rich_rrf': [RankedItem(item_id=1265, score=0.01639344262295082, ranks={'knn': 1}),
  RankedItem(item_id=41769, score=0.01639344262295082, ranks={'content': 1}),
  RankedItem(item_id=2243, score=0.016129032258064516, ranks={'knn': 2})]}

## 10. Offline Evaluation

Use the temporal holdout as relevant items. The metrics are intentionally
classic top-k metrics: precision, recall, hit rate, MRR, MAP, NDCG, coverage,
and diversity.

In [31]:
# Keep evaluation explicit and fast for a documentation notebook. The models
# were trained on the full train set; we evaluate the first 80 held-out users.
eval_users = test["user_id"].drop_duplicates().head(80)
test_sample = test[test["user_id"].isin(eval_users)].copy()

{
    "eval_users": test_sample["user_id"].nunique(),
    "eval_rows": len(test_sample),
}

{'eval_users': 80, 'eval_rows': 80}

In [32]:
popularity_eval = evaluate_recommender(
    bayesian_popularity,
    test_sample,
    user_column="user_id",
    item_column="item_id",
    k=10,
    catalog_items=catalog["item_id"],
    on_error="skip",
)

knn_eval = evaluate_recommender(
    item_knn,
    test_sample,
    user_column="user_id",
    item_column="item_id",
    k=10,
    catalog_items=catalog["item_id"],
    on_error="skip",
)

pd.DataFrame([popularity_eval, knn_eval], index=["popularity", "item_knn"]).T

,popularity,item_knn
users_evaluated,80.000000,80.000000
precision_at_k,0.006250,0.005000
recall_at_k,0.062500,0.050000
hit_rate_at_k,0.062500,0.050000
mrr_at_k,0.008800,0.005987
map_at_k,0.008800,0.005987
ndcg_at_k,0.020652,0.015486
catalog_coverage,0.006398,0.022833
users_skipped,0.000000,0.000000


In [33]:
# evaluate_rankings() works when recommendations are already materialized.
manual_recommendations = {
    user_id: bayesian_popularity.recommend(user_id, top_n=10)["item_id"].tolist()
    for user_id in test_sample["user_id"].drop_duplicates().head(10)
}
manual_relevant = {
    user_id: group["item_id"].drop_duplicates().tolist()
    for user_id, group in test_sample.groupby("user_id")
    if user_id in manual_recommendations
}

evaluate_rankings(
    manual_recommendations,
    manual_relevant,
    k=10,
    catalog_items=catalog["item_id"],
)

{'users_evaluated': 10,
 'precision_at_k': 0.0,
 'recall_at_k': 0.0,
 'hit_rate_at_k': 0.0,
 'mrr_at_k': 0.0,
 'map_at_k': 0.0,
 'ndcg_at_k': 0.0,
 'catalog_coverage': 0.0026472534745201853}

In [34]:
# Coverage asks: how much of the catalog appears in recommendations?
# Diversity asks: how different are the recommended items from each other?
coverage = catalog_coverage(
    manual_recommendations,
    catalog["item_id"],
    k=10,
)

diversity = intra_list_diversity(
    hybrid_weighted_recs["item_id"],
    catalog["item_id"],
    content_features,
    k=8,
)

{
    "catalog_coverage": coverage,
    "hybrid_intra_list_diversity": diversity,
}

{'catalog_coverage': 0.0026472534745201853,
 'hybrid_intra_list_diversity': 0.8461854159832001}

## 11. Practical Recipe

For a recommender project, MAna's workflow is:

1. Validate the interaction table and convert timestamps/scores.
2. Split by time, not random rows.
3. Build popularity first as the baseline and cold-start fallback.
4. Build content features when item metadata matters or new items arrive.
5. Build collaborative filtering when enough user-item history exists.
6. Fuse models when they solve different failure modes.
7. Evaluate ranking quality, coverage, and diversity together.

That is the point of `MAna.recommend`: not one magic recommender, but a set of
inspectable blocks that can be combined without breaking the data contract.